In [1]:
# bowaka_v2_lab notebook bootstrap cell — DO NOT EDIT BY HAND.
# Adds the lab's src/ (and its bowaka_common dependency) to sys.path and pins
# the working directory to the repo root, so `import bowaka_v2_lab` and
# repo-root-relative CONFIG_PATH parameters resolve identically under jupyter,
# papermill, and the QuantsLab scheduler.
import os
import sys
import time
from pathlib import Path

start_time = time.perf_counter()

_lab_root = None
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "bowaka_v2_lab" / "__init__.py").is_file():
        _lab_root = _candidate
        break
if _lab_root is None:
    raise RuntimeError(
        f"bowaka_v2_lab bootstrap: src/bowaka_v2_lab/ not found at or above {Path.cwd()}"
    )

# Pin CWD to the repo root so repo-root-relative CONFIG_PATH values resolve
# regardless of how the notebook was launched (jupyter CWD = notebook dir,
# scheduler = repo root). The lab ALWAYS lives at
# ``<repo_root>/research_notebooks/bowaka_v2_lab``, so derive the repo root from
# that canonical layout. A marker-only heuristic ("a dir with research_notebooks/
# AND Makefile") mis-matches the lab dir itself when it carries a Makefile and a
# stray nested research_notebooks/ — which then chdir's one level too deep and
# breaks every repo-root-relative path.
if _lab_root.parent.name == "research_notebooks":
    _repo_root = _lab_root.parent.parent
else:
    # Fallback: the repo root holds research_notebooks/bowaka_common (the sibling
    # package) — a marker a stray nested research_notebooks/ inside the lab lacks.
    _repo_root = _lab_root
    for _candidate in [_lab_root, *_lab_root.parents]:
        if (_candidate / "research_notebooks" / "bowaka_common").is_dir():
            _repo_root = _candidate
            break
os.chdir(_repo_root)

# Make the lab and its bowaka_common dependency importable from the working
# tree, even when the packages are not pip-installed. v1 bowaka_lab is
# deliberately excluded — v2 must not import v1.
for _src in (_lab_root / "src",
             _repo_root / "research_notebooks" / "bowaka_common" / "src"):
    if _src.is_dir() and str(_src) not in sys.path:
        sys.path.insert(0, str(_src))

import bowaka_v2_lab  # noqa: F401
print(f"bowaka_v2_lab {bowaka_v2_lab.__version__} (cwd={_repo_root})")


bowaka_v2_lab 0.1.0 (cwd=/quants-lab)


In [2]:
# Papermill parameters — override via `papermill -p <name> <value>` or in-place.
START_DATE = '2025-12-01'      # parity-window inclusive start (ISO date)
END_DATE   = '2026-05-23'      # parity-window inclusive end   (ISO date)
# Universe selection. Default mirrors what bowaka_v2 actually does live:
# build the PIT universe via the lab config's universe: criteria, screen
# to eligible_for_bowaka_equity_bucket, then monitor those symbols on
# both sides. Set SYMBOLS=[list] to override (debugging / unit-style runs);
# set MAX_UNIVERSE_SIZE=N to cap the screened result for fast smoke tests.
SYMBOLS = None
MAX_UNIVERSE_SIZE = None
PROD_CONFIG_PATH = 'research_notebooks/bowaka_v2_lab/reference/source_strategy/scripts/bowaka_v2_config.yaml'
LAB_CONFIG_PATH  = 'research_notebooks/bowaka_v2_lab/configs/bowaka_v2_actual_iex_current_code.yml'
LAKE_ROOT  = None              # default = bowaka_common.resolve_market_data_root()
COST_STRESS = 'base'           # passed to both sides identically; matches prod default
RUN_LABEL  = None              # default = UTC timestamp folder name
TIMEOUT_SEC = 1800             # production subprocess timeout (seconds, per session in chunked mode)
CHUNK_PER_SESSION = True       # True: per-session timing prints; False: single subprocess, canonical numerics
PARALLEL_WORKERS = 8           # >1: run session blocks across N worker subprocesses (implies chunked; identical output; capped at 16)
# Lake cache: on the ql-jupyter container the lake lives on a slow Docker
# host bind-mount (9p) where parallel workers stall in I/O-wait. Set
# LAKE_CACHE_DIR to a fast container-native path (e.g. '/opt/market_data_cache')
# to mirror the lake once and run every side off it (~10x faster). None = off.
LAKE_CACHE_DIR = '/opt/market_data_cache'     # fast local path to mirror the lake into (None = use LAKE_ROOT as-is)
FORCE_RECACHE = False          # re-copy even if the .lake_cache_complete marker is present


# 13 — Production-vs-Lab Parity

Empirical agreement between the production-side backtester
(`reference/source_strategy/scripts/bowaka_v2_backtest.py`) and the lab's
in-process backtester over a chosen window. Audit §14.5 thresholds drive
the stop-ship verdict; failing rows point at a specific divergence class.

**Mirrors what bowaka_v2 actually does live.** Universe screening runs first:
the lab's `build_pit_universe_for_sessions` resolves the PIT universe for
each session in the window, then `eligible_symbols(...)` reduces to the
survivors of the bowaka equity-bucket screen. With `CHUNK_PER_SESSION=True`
(default), EACH session's prod + lab are pointed at THAT session's eligible
survivors (one symbols file per session) — matching the live
screen-per-session flow and the lab's per-session PIT intersection (Phase 0
universe symmetry). The legacy window-union is available via
`per_session_universe=False`.

Override knobs: `SYMBOLS=[...]` to pass an explicit list (debugging);
`MAX_UNIVERSE_SIZE=N` to cap the screened universe to N symbols (fast smoke).

**Progress visibility.** With `CHUNK_PER_SESSION=True` (default) the runner
iterates session-by-session and prints `[i/N] DATE prod=Xs lab=Ys avg=... eta=...`
after each session, so you can see it's not hung. Trade-off: each lab session
starts at `initial_bankroll` (no carry-forward equity), so sizing-dependent
trade quantities can differ from the full-window run. Trade counts, entry/exit
times, and exit reasons are unaffected. For canonical numerics on a chosen
window, flip to `CHUNK_PER_SESSION=False`.

**Parallel sessions.** Set `PARALLEL_WORKERS=N` (>1) to run contiguous
session blocks across N worker subprocesses — **identical output**, faster
wall-clock on long multi-session windows (each worker warms its caches
once). Capped at 16 (the parity path opens no PostgreSQL connection); it
implies chunked mode.

**Lake cache (parallel speedup).** In the ql-jupyter container the shared
lake sits on a Docker host bind-mount (9p); parallel workers there stall in
I/O-wait (all workers `D`-state, ~1 core) instead of using CPU. Set
`LAKE_CACHE_DIR='/opt/market_data_cache'` to mirror the lake once onto the
container-native filesystem and run every side off it — measured ~10x faster
(bind 1102s -> cache 108s on a 20-session / 40-symbol block), byte-identical
results. The copy is one-time (a `.lake_cache_complete` marker guards reuse);
`FORCE_RECACHE=True` refreshes it. Note: worker count past ~4-8 gives little
extra (a 20-session window oversubscribes 16 workers + their prod
subprocesses). Requires the `run_lab_backtester` lake-root fix — pre-fix the
lab side ignored `LAKE_ROOT` / `LAKE_CACHE_DIR` and always read the bind-mount.

**Requires Phase 0's fix landed** — pre-fix the production side always read
deterministic synthetic data and the parity metrics are meaningless.

In [3]:
# Resolve lake root + bowaka-v2 universe screen.
import datetime as _dt
from pathlib import Path

from bowaka_common.marketdata.store import resolve_market_data_root
from bowaka_v2_lab.parity import build_parity_universe

_lake_root = Path(LAKE_ROOT).resolve() if LAKE_ROOT else resolve_market_data_root(None, create=False)
print(f'lake_root: {_lake_root}')

# Optional one-time mirror of the lake onto a fast filesystem (dodges the
# Docker 9p bind-mount that serializes parallel workers into I/O-wait).
if LAKE_CACHE_DIR:
    import shutil, time as _time
    _cache_dir = Path(LAKE_CACHE_DIR)
    _marker = _cache_dir / '.lake_cache_complete'
    if _marker.is_file() and not FORCE_RECACHE:
        print(f'lake cache present at {_cache_dir} - reusing (FORCE_RECACHE=True to re-copy)')
    else:
        print(f'priming lake cache: {_lake_root} -> {_cache_dir} (one-time; reads bind-mount once)...')
        _t0 = _time.monotonic()
        shutil.copytree(_lake_root, _cache_dir, dirs_exist_ok=True)
        _marker.write_text('ok', encoding='utf-8')
        print(f'lake cache primed in {_time.monotonic() - _t0:.0f}s')
    _lake_root = _cache_dir
    print(f'effective lake_root -> {_lake_root} (cached; workers avoid the 9p bind-mount)')

_start = _dt.date.fromisoformat(START_DATE)
_end   = _dt.date.fromisoformat(END_DATE)

if SYMBOLS is not None:
    _syms = [str(s) for s in SYMBOLS]
    _src  = 'explicit'
else:
    _syms = build_parity_universe(
        start_date=_start, end_date=_end,
        lab_config_path=Path(LAB_CONFIG_PATH),
        lake_root=_lake_root,
        max_universe_size=MAX_UNIVERSE_SIZE,
    )
    _src = ('pit_screen_capped' if MAX_UNIVERSE_SIZE else 'pit_screen')
if not _syms:
    raise RuntimeError('parity universe is empty after screening — check the window and lab config')
print(f'universe ({_src}): {len(_syms)} symbols; head={_syms[:5]} tail={_syms[-5:]}')
print(f'window:   {_start} -> {_end}')


lake_root: /quants-lab/research_notebooks/market_data
priming lake cache: /quants-lab/research_notebooks/market_data -> /opt/market_data_cache (one-time; reads bind-mount once)...
lake cache primed in 1235s
effective lake_root -> /opt/market_data_cache (cached; workers avoid the 9p bind-mount)
universe (pit_screen): 1240 symbols; head=['AAL', 'ABAT', 'ABCL', 'ABEO', 'ABEV'] tail=['ZIM', 'ZIP', 'ZNTL', 'ZURA', 'ZVRA']
window:   2025-12-01 -> 2026-05-23


In [4]:
# Pre-flight workload estimate. Lets you cancel a run that's about to
# eat hours before kicking off both subprocesses.
import exchange_calendars as _xcals
import pandas as _pd

_cal = _xcals.get_calendar('XNYS')
_sessions = [_pd.Timestamp(s).date() for s in _cal.sessions_in_range(
    _pd.Timestamp(_start), _pd.Timestamp(_end))] or [_start]
_n_symdays = len(_syms) * len(_sessions)
print(f'sessions:      {len(_sessions)} XNYS days')
print(f'symbols:       {len(_syms)} ({_src})')
print(f'symbol-days:   {_n_symdays:,}  (prod + lab each scan this many)')
print(f'timeout:       {TIMEOUT_SEC}s per side')
if _n_symdays > 5_000:
    print()
    print(f'WARNING: {_n_symdays:,} symbol-days is a real-universe parity run.')
    print(f'  - first runs: set MAX_UNIVERSE_SIZE=30 (cap to 30 syms) or pin SYMBOLS=[...].')
    print(f'  - real runs:  expect minutes to ~1 hour; bump TIMEOUT_SEC if needed.')
    print(f'  - progress:   tail -f <run_root>/production/production.stderr.log')


sessions:      120 XNYS days
symbols:       1240 (pit_screen)
symbol-days:   148,800  (prod + lab each scan this many)
timeout:       1800s per side

  - first runs: set MAX_UNIVERSE_SIZE=30 (cap to 30 syms) or pin SYMBOLS=[...].
  - real runs:  expect minutes to ~1 hour; bump TIMEOUT_SEC if needed.
  - progress:   tail -f <run_root>/production/production.stderr.log


In [5]:
# Run both sides + compute parity.
from bowaka_v2_lab.parity import run_parity, render_markdown_report

_label = RUN_LABEL or _dt.datetime.now(_dt.UTC).strftime('%Y%m%dT%H%M%SZ')
_run_root = Path('research_notebooks/bowaka_v2_lab/artifacts/parity/lab_vs_production') / _label
_run_root.mkdir(parents=True, exist_ok=True)
print(f'run_root:      {_run_root}')
print(f'progress log:  {_run_root}/production/production.stderr.log')

report = run_parity(
    start_date=_start, end_date=_end,
    symbols=_syms,
    prod_config_path=Path(PROD_CONFIG_PATH),
    lab_config_path=Path(LAB_CONFIG_PATH),
    lake_root=_lake_root,
    cost_stress=COST_STRESS,
    run_root=_run_root,
    timeout_sec=int(TIMEOUT_SEC),
    chunk_per_session=bool(CHUNK_PER_SESSION) or int(PARALLEL_WORKERS) > 1,
    parallel_workers=int(PARALLEL_WORKERS),
)
print(f'prod_n_trades={report.prod_n_trades}  lab_n_trades={report.lab_n_trades}')
print(f'trade_intersection_rate={report.trade_intersection_rate:.4f}')
print(f'fill_price_mae_bps={report.fill_price_mae_bps:.4f}')
print(f'passes_audit_thresholds={report.passes_audit_thresholds}')
if report.failing_metrics:
    print(f'failing metrics: {report.failing_metrics}')


run_root:      research_notebooks/bowaka_v2_lab/artifacts/parity/lab_vs_production/20260602T061929Z
progress log:  research_notebooks/bowaka_v2_lab/artifacts/parity/lab_vs_production/20260602T061929Z/production/production.stderr.log
[parity] parallel mode: 120 sessions across 8 worker(s) (cap=16)
prod_n_trades=1090  lab_n_trades=884
trade_intersection_rate=0.0051
fill_price_mae_bps=28.6201
passes_audit_thresholds=False
failing metrics: ['trade_intersection_rate', 'fill_price_mae_bps', 'exit_reason_match_rate', 'daily_pnl_sign_match_rate']


In [6]:
# Persist the paste-back Markdown.
_md_path = _run_root / 'parity_report.md'
render_markdown_report(report, output_path=_md_path)
print(f'wrote: {_md_path}')
print()
print(_md_path.read_text(encoding='utf-8'))

end_time = time.perf_counter()
execution_time = end_time - start_time
print(f"Script finished in {execution_time:.4f} seconds.")

execution_minutes = execution_time / 60
print(f"Script finished in {execution_minutes:.2f} minutes.")

wrote: research_notebooks/bowaka_v2_lab/artifacts/parity/lab_vs_production/20260602T061929Z/parity_report.md

# Production-vs-lab parity report

- generated_at: 2026-06-02T06:45:43.999537+00:00

## Window

| Field | Value |
|---|---|
| window_start | 2025-12-01 |
| window_end | 2026-05-23 |
| universe_size | 1240 |
| n_sessions | 120 |
| n_trade_sessions | 118 |

## Counts

| | Production | Lab |
|---|---:|---:|
| candidates | 0 | 5419 |
| trades     | 1090 | 884 |
| gross_pnl  | -10229.43 | 214.38 |

## Parity metrics

| Metric | Actual | Threshold | Result |
|---|---:|---:|:--:|
| candidate_recall | N/A | ≥ 0.9900 | N/A |
| gate_match_rate | N/A | ≥ 0.9500 | N/A |
| trade_intersection_rate | 0.0051 | ≥ 0.9000 | FAIL |
| fill_price_mae_bps | 28.6201 | ≤ 5.0000 | FAIL |
| exit_reason_match_rate | 0.6000 | ≥ 0.9000 | FAIL |
| daily_pnl_sign_match_rate | 0.5932 | ≥ 0.9500 | FAIL |

## Trades only in production

| session_date | symbol | entry_ts_minute | entry_price | qty | exit_reason |